In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import require_widget
from lib.decertification import (
    detect_decertification_candidates,
    write_decertification_proposals,
)
dbutils.widgets.text("catalog",         "")
dbutils.widgets.text("control_schema",  "uc_hygiene")
dbutils.widgets.text("staleness_days",  "30")
dbutils.widgets.text("dry_run",         "false")

catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
staleness_days = int(dbutils.widgets.get("staleness_days") or "30")
dry_run        = dbutils.widgets.get("dry_run").lower() == "true"

print(f"Control: {catalog}.{control_schema}")
print(f"Staleness threshold: {staleness_days}d")
print(f"Dry run: {dry_run}")
import time as _t; _task_start = _t.time()


In [0]:
# Detect certified tables that have drifted, gone stale, or lost features
candidates = detect_decertification_candidates(
    spark, catalog, control_schema, staleness_days
)

candidate_count = candidates.count()
print(f"Decertification candidates: {candidate_count}")
if candidate_count > 0:
    candidates.groupBy("decert_reason", "decert_severity").count().show()


In [0]:
# Write proposals (respects dry_run)
result = write_decertification_proposals(
    spark, candidates, catalog, control_schema, dry_run=dry_run
)


In [0]:
_duration = int(_t.time() - _task_start)
print(f"\n{'='*52}")
print(f"  DECERTIFICATION DETECTION COMPLETE")
print(f"{'='*52}")
print(f"  Candidates found: {candidate_count}")
print(f"  Dry run:          {dry_run}")
print(f"  Duration:         {_duration}s")
print(f"{'='*52}")

# Write execution record
from datetime import date
try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      DATE('{date.today()}'),
      'uc_steward_daily_governance',
      'p3_decertification',
      'p3_remediation',
      'success',
      {candidate_count},
      {candidate_count},
      {candidate_count if not dry_run else 0},
      {_duration},
      'dry_run={dry_run}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")
